In [1]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3" / "2007_2008"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2007_2008_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2007_2008_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2007_2008_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2007_2008_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2007_2008_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2007_2008_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (10149, 45)
Labs: (9307, 35)
Questionnaire: (10149, 455)
Examination: (9762, 23)
Dietary: (9762, 242)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,10149,45,10149,0
1,labs,9307,35,9307,0
2,questionnaire,10149,455,10149,0
3,Examination,9762,23,9762,0
4,Dietary,9762,242,9762,0


📊 Final Unified Dataset Shape: (10149, 796)
Unique SEQN Count: 10149
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,DMQMILIT,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWS
0,41475,5,2,2,2,62,751,752,5,2,...,0.0,0.16,0.008,0.003,0.026,2,725.8199999999999,725.8199999999999,0.0,2
1,41476,5,2,1,2,6,81,82,5,,...,0.021,0.036000000000000004,0.0,0.0,0.005,2,340.69,103.69,237.0,1
2,41477,5,2,2,1,71,859,860,3,1,...,0.047,0.29900000000000004,0.268,0.13,0.33999999999999997,1,0.0,0.0,0.0,3
3,41478,5,2,2,2,1,17,17,3,,...,,,,,,,,,,
4,41479,5,2,1,1,52,629,630,1,2,...,0.0,0.062,0.004,0.002,0.001,2,474.0,0.0,474.0,99
5,41480,5,2,1,1,6,81,81,1,,...,0.0,0.024,0.003,0.006,0.009,3,355.5,355.5,0.0,1
6,41481,5,2,1,1,21,254,254,4,2,...,0.0,0.214,0.02,0.006,0.0,3,0.0,0.0,0.0,1
7,41482,5,2,2,1,64,778,779,1,2,...,0.0,0.247,0.011,0.021,0.055999999999999994,3,474.0,0.0,474.0,1
8,41483,5,2,1,1,66,803,804,4,1,...,0.069,0.057,0.0,0.002,0.0,2,237.0,237.0,0.0,1
9,41484,5,2,1,1,0,9,10,3,,...,0.0,0.0,0.0,0.0,0.0,2,0.0,0.0,0.0,4



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2007_2008_corrected.csv
